In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [1]:
import sys
import matplotlib.pyplot as plt

from spice import SpiceEstimator, csv_to_dataset, plot_session, split_data_along_sessiondim, BaseModel
from spice.precoded import workingmemory

sys.path.append('../../..')
from weinhardt2026.utils.benchmarking_gru import Model as GRU, training
from weinhardt2026.studies.dezfouli2019.dezfouli2019 import GQLModel

from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals

# For custom RNN
import torch

## Load dataset

Let's load the data first with the `csv_to_dataset` method. This method returns a `SpiceDataset` object which we can use right away 

In [2]:
# Load your data
path_data = 'data/dezfouli2019.csv'
dataset = csv_to_dataset(file = path_data)

# structure of dataset:
# dataset has two main attributes: xs -> inputs; ys -> targets (next action)
# shape: (n_participants*n_blocks*n_experiments, n_trials, n_timesteps=1, features)
# features are (n_actions * action, n_actions * reward, n_additional_inputs * additional_input, timestep, trial, block, experiment, participant)

# in order to set up the participant embedding we have to compute the number of unique participants in our data 
# to get the number of participants n_participants we do:
n_participants = len(dataset.xs[..., -1].unique())
n_actions = dataset.ys.shape[-1]

print(f"Shape of dataset: {dataset.xs.shape}")
print(f"Number of participants: {n_participants}")
print(f"Number of actions in dataset: {n_actions}")

Shape of dataset: torch.Size([1212, 202, 1, 9])
Number of participants: 101
Number of actions in dataset: 2


In [3]:
test_sessions = (3, 6, 9)
dataset_train, dataset_test = split_data_along_sessiondim(dataset, test_sessions)

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [ ]:
# fitting without SINDy coefficients

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=workingmemory.SpiceModel,
        spice_config=workingmemory.CONFIG,
        n_actions=n_actions,
        n_participants=n_participants,
        
        epochs=1000,
        warmup_steps=200,
        sindy_weight=0,
        
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [4]:
path_spice = 'params/spice_dezfouli2019.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=workingmemory.SpiceModel,
        spice_config=workingmemory.CONFIG,
        n_actions=2,
        n_participants=n_participants,
        kwargs_spice_class={'reward_binary': True},
        
        epochs=1000,
        warmup_steps=200,
        ensemble_size=10,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.01,
        sindy_ensemble_pruning=0.05,
        sindy_library_polynomial_degree=2,
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        verbose=True,
        save_path_spice=path_spice,
    )

In [ ]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_train.xs, dataset_train.ys)

In [ ]:
estimator.load_spice(path_spice)
estimator.set_device(torch.device('cpu'))

In [ ]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=0)

## Benchmarking

### Generalized Q-Learning Model by Dezfouli (2019)

In [ ]:
# 1. stick to low effort 
# 2. two sets of params for low and high effort

gql = GQLModel(
    n_participants=n_participants,
    batch_first=True,
    )

path_gql = path_spice.replace('spice_', 'gql_')

In [ ]:
epochs = 1000
optimizer = torch.optim.Adam(params=gql.parameters(), lr=0.01)

gql = training(
    model=gql,
    optimizer=optimizer,
    epochs=epochs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    device=torch.device('cpu'),
)

torch.save(gql.state_dict(), path_gql)

In [ ]:
gql.load_state_dict(torch.load(path_gql, map_location='cpu'))

### GRU Model

In [ ]:
gru = GRU(n_actions)
path_gru = path_spice.replace('spice_', 'gru_')

In [ ]:
epochs = 1000
optimizer = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

In [ ]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))
gru.eval()

# ANALYSIS

## Analysis model evaluation

In [ ]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=gql.to(torch.device('cpu')),
    gru_model=gru.to(torch.device('cpu')),
)

## Analysis coefficient distributions

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

## Analysis Individual Differences

In [ ]:
analysis_coefficients_individuals(
    criterion="diag",
    analysis="disc",  # also: "cont"
    reference="Control",  # only necessary if analysis="disct"
    
    path_data=path_data,
    
    spice_model=estimator,
    
    dir_output='results',
)

# EXPERIMENT EMBEDDING

## Dataset setup

In [5]:
path_data = 'data/dezfouli2019_experimentid.csv'
dataset = csv_to_dataset(file = path_data)

n_actions = dataset.ys.shape[-1]
n_participants = dataset.xs[..., -1].unique().shape[0]
n_experiments = dataset.xs[..., -2].unique().shape[0]
print(f"Number of participants: {n_participants}")
print(f"Number of experiments: {n_experiments}")

test_sessions = (3, 6, 9)
dataset_train, dataset_test = split_data_along_sessiondim(dataset, test_sessions)

Number of participants: 101
Number of experiments: 3


## Spice Setup

In [6]:
class SpiceEmbModel(BaseModel):
    """
    Working memory as explicit buffer of recent rewards.
    
    Key difference from value learning:
    - Stores individual past rewards (not aggregated statistics)
    - Fixed capacity (buffer size)
    - Perfect memory for items in buffer
    - Items fall out of buffer (discrete forgetting)
    """
    
    def __init__(self, reward_binary: bool = False, **kwargs):
        super().__init__(**kwargs)
        
        dropout = 0.1
        
        self.participant_embedding = self.setup_embedding(self.n_participants, self.embedding_size, dropout=dropout)
        self.experiment_embedding = self.setup_embedding(self.n_experiments, 2, dropout=dropout)
        
        # Value learning module (slow updates)
        # Can use recent reward history to modulate learning
        self.setup_module(key_module='value_reward_chosen', input_size=4+self.embedding_size+2, dropout=dropout)  # -> 21 terms
        self.setup_module(key_module='value_reward_not_chosen', input_size=3+self.embedding_size+2, dropout=dropout)  # -> 21 terms
        
        # self.setup_module(key_module='value_choice', input_size=4+self.embedding_size, dropout=dropout, include_bias=True) # -> 21 terms; bias not necessary when module is applied equally to all options
        self.setup_module(key_module='value_choice_chosen', input_size=3+self.embedding_size+2, dropout=dropout) # -> 21 terms; bias not necessary when module is applied equally to all options
        self.setup_module(key_module='value_choice_not_chosen', input_size=3+self.embedding_size+2, dropout=dropout) # -> 21 terms; bias not necessary when module is applied equally to all options
        
        self.preprocess_coefficients(reward_binary=reward_binary)
        
    def forward(self, inputs, prev_state=None, batch_first=False):
        spice_signals = self.init_forward_pass(inputs, prev_state, batch_first)

        # perform time-invariant computations
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        experiment_embedding = self.experiment_embedding(spice_signals.experiment_ids)

        for trial in spice_signals.trials:
            
            # REWARD VALUE UPDATES
            self.call_module(
                key_module='value_reward_chosen',
                key_state='value_reward',
                action_mask=spice_signals.actions[trial],
                inputs=(
                    spice_signals.rewards[trial],
                    self.state['buffer_reward_1'],
                    self.state['buffer_reward_2'],
                    self.state['buffer_reward_3'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
                experiment_index=spice_signals.experiment_ids,
                experiment_embedding=experiment_embedding,
            )

            self.call_module(
                key_module='value_reward_not_chosen',
                key_state='value_reward',
                action_mask=1-spice_signals.actions[trial],
                inputs=(
                    self.state['buffer_reward_1'],
                    self.state['buffer_reward_2'],
                    self.state['buffer_reward_3'],
                    ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
                experiment_index=spice_signals.experiment_ids,
                experiment_embedding=experiment_embedding,
            )

            # CHOICE VALUE UPDATES
            # self.call_module(
            #     key_module='value_choice',
            #     key_state='value_choice',
            #     action_mask=None,
            #     inputs=(
            #         spice_signals.actions[trial],
            #         self.state['buffer_action_1'],
            #         self.state['buffer_action_2'],
            #         self.state['buffer_action_3'],
            #     ),
            #     participant_index=spice_signals.participant_ids,
            #     participant_embedding=participant_embedding,
            #     experiment_index=spice_signals.experiment_ids,
            #     experiment_embedding=experiment_embedding,
            # )
            
            self.call_module(
                key_module='value_choice_chosen',
                key_state='value_choice',
                action_mask=spice_signals.actions[trial],
                inputs=(
                    self.state['buffer_action_1'],
                    self.state['buffer_action_2'],
                    self.state['buffer_action_3'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
                experiment_index=spice_signals.experiment_ids,
                experiment_embedding=experiment_embedding,
            )
            self.call_module(
                key_module='value_choice_not_chosen',
                key_state='value_choice',
                action_mask=1-spice_signals.actions[trial],
                inputs=(
                    self.state['buffer_action_1'],
                    self.state['buffer_action_2'],
                    self.state['buffer_action_3'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
                experiment_index=spice_signals.experiment_ids,
                experiment_embedding=experiment_embedding,
            )

            # BUFFER UPDATES:
            # REWARD BUFFER UPDATES: Shift reward buffer for chosen action, keep for not chosen action
            # ACTION BUFFER UPDATES: Shift all buffer entries according to action
            self.state['buffer_reward_3'] = self.state['buffer_reward_2'] * spice_signals.actions[trial] + self.state['buffer_reward_3'] * (1-spice_signals.actions[trial])
            self.state['buffer_reward_2'] = self.state['buffer_reward_1'] * spice_signals.actions[trial] + self.state['buffer_reward_2'] * (1-spice_signals.actions[trial])
            self.state['buffer_reward_1'] = torch.where(spice_signals.actions[trial]==1, spice_signals.rewards[trial], 0) + torch.where(spice_signals.actions[trial]==0, self.state['buffer_reward_1'], 0)
            self.state['buffer_action_3'] = self.state['buffer_action_2']
            self.state['buffer_action_2'] = self.state['buffer_action_1']
            self.state['buffer_action_1'] = spice_signals.actions[trial]

            # compute logits for current timestep
            spice_signals.logits[trial] = self.state['value_reward'] + self.state['value_choice']

        
        spice_signals = self.post_forward_pass(spice_signals, batch_first)

        return spice_signals.logits, self.get_state()
    
    def preprocess_coefficients(self, reward_binary: bool = True):
        # remove unnecessary candidate terms, e.g. polynomials of binary signals
        # if reward_binary: reward[t] = reward[t]^2 -> presence[reward[t]^2] = 0
        # accounts for ALL control signals in workingmemory model if reward is binary; else only choice signals
        
        candidate_terms = self.get_candidate_terms()
        for module in self.get_modules():
            if ('reward' in module and reward_binary) or 'choice' in module:
                control_signals = self.spice_config.library_setup[module]
                for cs in control_signals:
                    for ict, ct in enumerate(candidate_terms[module]):
                        if cs+'^' in ct:
                            self.sindy_coefficients_presence[module][..., ict] = 0
        


In [7]:
path_spice_emb = path_spice.replace('spice_', 'spice_emb_')

estimator_emb = SpiceEstimator(
    spice_class=SpiceEmbModel,
    spice_config=workingmemory.CONFIG,
    n_actions=n_actions,
    n_participants=n_participants,
    n_experiments=n_experiments,
    kwargs_spice_class={'reward_binary': True},
    
    epochs=1000,
    warmup_steps=200,
        
    ensemble_size=10,
    sindy_weight=0.1,
    sindy_alpha=0.0001,
    sindy_pruning_frequency=100,
    sindy_ensemble_pruning=0.05,
    sindy_threshold_pruning=0.01,
    sindy_library_polynomial_degree=2,
    
    save_path_spice=path_spice_emb,
    verbose=True,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    )

In [ ]:
estimator_emb.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

 14%|█████                                | 136/1000 [18:07<1:55:10,  8.00s/it, L(Train)=0.3032621, L(Val,RNN)=0.2915086, L(Val,SINDy)=0.4212787, Conv=1.47e-04]
----------------------------------------------------------------------------------------------------------------------------------------------------------------
SPICE Model (Coefficients: 53):
value_reward_chosen[t+1]     = 0.113 1 + 0.712 value_reward_chosen[t] + -1.24 reward[t] + 0.714 reward[t-1] + 0.409 reward[t-2] + 0.241 reward[t-3] + 0.036 value_reward_chosen^2 + 0.02 value_reward_chosen*reward[t] + -0.087 value_reward_chosen*reward[t-1] + 0.003 value_reward_chosen*reward[t-2] + 0.053 value_reward_chosen*reward[t-3] + 0.025 reward[t]*reward[t-1] + 0.063 reward[t]*reward[t-2] + 0.139 reward[t]*reward[t-3] + -0.035 reward[t-1]*reward[t-2] + -0.069 reward[t-1]*reward[t-3] + -0.065 reward[t-2]*reward[t-3] 
value_reward_not_chosen[t+1] = -0.053 1 + 0.809 value_reward_not_chosen[t] + -0.056 reward[t-1] + 0.003 reward[t-2] + 0.

In [ ]:
estimator_emb.load_spice(path_spice_emb)

## Analysis

### Analysis model comparison

In [ ]:
# SPICE with experiment embedding
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator_emb,
)